# Tune classical ADMM on Colab

Before running, choose **Runtime > Change runtime type > GPU**. The solver is a stack of FFTs, so a GPU is many times faster than the laptop CPU.

Run folders are saved to Google Drive, so they survive a disconnect, and a rerun of the sweep skips the grid points that already finished.

## 1. Check the GPU and mount Drive

In [ ]:
!nvidia-smi -L

from google.colab import drive
drive.mount("/content/drive")

EXPERIMENTS_ROOT = "/content/drive/MyDrive/ece488/experiments"
!mkdir -p {EXPERIMENTS_ROOT}

## 2. Get the code

In [ ]:
import os
from pathlib import Path

BRANCH = "main"
REPO_DIR = Path("/content/ECE488")

if not REPO_DIR.exists():
    !git clone --quiet --branch {BRANCH} https://github.com/olivcho/ECE488.git {REPO_DIR}

os.chdir(REPO_DIR)
!git pull --quiet --ff-only
!git log -1 --format='%h %s'
!git status --short

## 3. Install dependencies

This keeps Colab's own PyTorch so it matches the GPU drivers. Each run records the versions it used.

In [ ]:
%pip install -q lpips torchmetrics datasets huggingface_hub pyyaml pytest tifffile
%pip install -q --no-deps -e .

# A first local download stalled on Hugging Face's Xet path, so use plain HTTPS.
os.environ["HF_HUB_DISABLE_XET"] = "1"

!python -m pytest -q

## 4. Short test

This reconstructs 8 images from the first shard with 100 iterations. Check that the preview looks sensible, then use the printed seconds per image at iteration 100 to estimate the sweep:

`sweep seconds ≈ seconds per image × 2 × 200 images × 9 grid points`

In [ ]:
!python scripts/run_admm.py --config configs/admm_quick.yaml --experiments-root {EXPERIMENTS_ROOT}

## 5. Sweep

This downloads all 7.6 GB, because the validation images are drawn from every shard, so the split matches the U-Net's. It then runs 200 iterations for each of the 9 (τ, μ) grid points on 200 validation images. Each point becomes its own run folder under `admm_sweep/`. The test split is never used.

In [ ]:
!python scripts/sweep_admm.py --config configs/admm_sweep.yaml --experiments-root {EXPERIMENTS_ROOT}

## 6. Resume after a disconnect

Run sections 1 to 3 again, then run the sweep cell again. Finished grid points are skipped, and an interrupted point starts over as a new version.

## 7. See the results

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(f"{EXPERIMENTS_ROOT}/admm_sweep/summary.csv")
final = summary[summary.iteration == summary.iteration.max()]
display(final.pivot(index="mu1", columns="tau", values="val_screen_psnr").round(2))
display(summary.groupby("iteration")["val_screen_psnr"].max().round(2))

best = final.loc[final.val_screen_psnr.idxmax()]
print(f"best: run {best.run}, tau={best.tau}, mu={best.mu1}, screen PSNR {best.val_screen_psnr:.2f} dB")
display(Image(str(sorted(Path(EXPERIMENTS_ROOT, "admm_sweep", best.run, "previews").glob("*.png"))[-1])))

## 8. Baseline on the full validation and test splits

The best setting runs on all 1,000 validation images, then on the 999 test images. These are the same images the U-Net and Le-ADMM report on, chosen by the shared split helper. The overrides are written into each run's `config.yaml`, so the result stays traceable. The test number is final: do not tune on it.

In [ ]:
BEST = (f"--set admm.tau={best.tau} --set admm.mu1={best.mu1} --set admm.mu2={best.mu1} --set admm.mu3={best.mu1} "
        f"--set admm.iterations={int(best.iteration)}")
!python scripts/run_admm.py --config configs/admm.yaml --experiments-root {EXPERIMENTS_ROOT} {BEST} --set data.split=val
!python scripts/run_admm.py --config configs/admm.yaml --experiments-root {EXPERIMENTS_ROOT} {BEST} --set data.split=test

## 9. Optional: score the training rows

The U-Net and Le-ADMM train on the first 5,000 training rows. Scoring ADMM on those same rows shows how much of a learned model's training-set score is memorization, since ADMM cannot memorize.

In [ ]:
!python scripts/run_admm.py --config configs/admm.yaml --experiments-root {EXPERIMENTS_ROOT} {BEST} --set data.split=train --set data.max_examples=5000